In [ ]:

"""
Span-level CER Dataset for Hindi Text Normalization Evaluation
==============================================================

JSONL format (one JSON per line):
    tagged_output : str   — sentence with spans like <raw_span><TAG>
    norm          : str   — same structure but spans are GT normalized
    mt5           : str   — model output (plain, no tags)
    hints_ib      : str   — model output (plain, no tags)
    rb            : str   — model output (plain, no tags)

__getitem__ returns a dict with:
    sentence_id        : int
    unnorm_sentence    : str   — plain text of tagged_output (tags stripped)
    spans              : list of span-dicts, each containing:
        span_index     : int
        tag            : str
        raw_span       : str   — unnormalized span text
        gt_span        : str   — GT normalized span text
        left_context   : str
        right_context  : str
        model_spans    : dict  model_name → extracted span text (or None)
        cer            : dict  model_name → CER float
"""

import json
import re
from pathlib import Path
from typing import Optional, List, Dict, Tuple, Any
import jiwer
from torch.utils.data import Dataset,DataLoader
from transformers import MT5EncoderModel, MT5Tokenizer
tokenizer = MT5Tokenizer.from_pretrained("google/mt5-small") 

# ─────────────────────────────────────────────────────────────────────────────
# Constants
# ─────────────────────────────────────────────────────────────────────────────

MODEL_KEYS = ["mt5", "hints_ib", "rb"]


# ─────────────────────────────────────────────────────────────────────────────
# Regex: matches  <any text><UPPERCASE_TAG>
# Handles multi-char Unicode spans (Hindi, digits, punctuation, spaces)
# ─────────────────────────────────────────────────────────────────────────────
_SPAN_RE = re.compile(r'<([^<>]+)><([A-Z][A-Z0-9_]*)>')


# ─────────────────────────────────────────────────────────────────────────────
# Parsing helpers
# ─────────────────────────────────────────────────────────────────────────────

def _strip_tags(tagged: str) -> str:
    """Remove all <span><TAG> markup and return plain text."""
    return _SPAN_RE.sub(lambda m: m.group(1), tagged).strip()


def _parse_tagged(tagged: str) -> List[Dict[str, Any]]:
    """
    Split a tagged string into alternating text / span parts.

    Returns a list of dicts:
        { 'kind': 'text'|'span', 'text': str, 'tag': str|None }
    """
    result = []
    cursor = 0
    for m in _SPAN_RE.finditer(tagged):
        if m.start() > cursor:
            result.append({'kind': 'text', 'text': tagged[cursor:m.start()], 'tag': None})
        result.append({'kind': 'span', 'text': m.group(1), 'tag': m.group(2)})
        cursor = m.end()
    if cursor < len(tagged):
        result.append({'kind': 'text', 'text': tagged[cursor:], 'tag': None})
    return result


def _build_context(parts: List[Dict[str, Any]], span_idx: int, n: int) -> Tuple[str, str]:
    """
    Return (left_context, right_context) — up to n words each — for the span
    at position `span_idx` in `parts`.  Only plain 'text' chunks contribute.
    """
    left_text = ''.join(p['text'] for p in parts[:span_idx] if p['kind'] == 'text')
    right_text = ''.join(p['text'] for p in parts[span_idx+1:] if p['kind'] == 'text')

    left_words = left_text.split()
    right_words = right_text.split()

    return (
        ' '.join(left_words[-n:]) if left_words else '',
        ' '.join(right_words[:n]) if right_words else '',
    )


def extract_raw_spans(tagged: str, context_words: int = 3) -> List[Dict[str, str]]:
    """
    Parse tagged_output.

    Returns list of dicts:
        raw_span, tag, left_context, right_context
    """
    parts = _parse_tagged(tagged)
    spans = []
    for i, p in enumerate(parts):
        if p['kind'] != 'span':
            continue
        left_ctx, right_ctx = _build_context(parts, i, context_words)
        spans.append({
            'raw_span': p['text'],
            'tag': p['tag'],
            'left_context': left_ctx,
            'right_context': right_ctx,
        })
    return spans


def extract_gt_spans(norm: str) -> List[str]:
    """
    Parse the `norm` field.

    Returns list of GT normalized span strings in the same order as
    tagged_output spans.
    """
    return [m.group(1) for m in _SPAN_RE.finditer(norm)]


# ─────────────────────────────────────────────────────────────────────────────
# Context-based span extraction from a plain model output
# ─────────────────────────────────────────────────────────────────────────────

def _find_between(text: str, left_ctx: str, right_ctx: str) -> Optional[str]:
    """
    Extract the substring of `text` that lies between `left_ctx` and
    `right_ctx`.

    Degrades gracefully: if the full context string isn't found verbatim,
    progressively shorter suffixes (left) and prefixes (right) are tried.
    This handles cases where the model slightly rephrased surrounding words.
    """

    def after(haystack: str, needle: str) -> int:
        """Index just AFTER `needle` in `haystack`, or -1."""
        if not needle:
            return 0
        idx = haystack.find(needle)
        return (idx + len(needle)) if idx != -1 else -1

    def before(haystack: str, needle: str) -> int:
        """Start index of `needle` in `haystack`, or -1."""
        if not needle:
            return len(haystack)
        return haystack.find(needle)

    left_words = left_ctx.split()
    right_words = right_ctx.split()

    for lw in range(len(left_words), -1, -1):
        lctx = ' '.join(left_words[-lw:]) if lw else ''
        pos = after(text, lctx)
        if pos == -1:
            continue

        for rw in range(len(right_words), -1, -1):
            rctx = ' '.join(right_words[:rw]) if rw else ''
            region = text[pos:]
            rpos = before(region, rctx)
            if rpos == -1:
                continue

            extracted = region[:rpos].strip()
            if extracted or (not lctx and not rctx):
                return extracted

    return None


# ─────────────────────────────────────────────────────────────────────────────
# CER
# ─────────────────────────────────────────────────────────────────────────────

_cer_transform = jiwer.Compose([
    jiwer.RemoveMultipleSpaces(),
    jiwer.Strip(),
    jiwer.ReduceToListOfListOfChars(),
])


def compute_cer(hypothesis: str, reference: str) -> float:
    """Character Error Rate in [0, 1]."""
    if not reference:
        return 0.0 if not hypothesis else 1.0
    if not hypothesis:
        return 1.0
    return jiwer.cer(
        reference,
        hypothesis,
        reference_transform=_cer_transform,
        hypothesis_transform=_cer_transform,
    )


# ─────────────────────────────────────────────────────────────────────────────
# Dataset
# ─────────────────────────────────────────────────────────────────────────────

class SpanCERDataset(Dataset):
    """
    Loads a JSONL file and exposes per-sentence span data.

    Each line in the JSONL must have:
        tagged_output, norm, mt5, hints_ib, rb

    Parameters
    ----------
    filepath      : path to the .jsonl file
    model_keys    : model field names in the JSONL  (default: mt5, hints_ib, rb)
    context_words : number of surrounding words used as context anchors
    skip_on_error : whether to skip problematic sentences (default: True)

    __getitem__(i) returns:
    {
        "sentence_id"     : int,
        "unnorm_sentence" : str,          # plain text, no tags
        "spans": [
            {
                "span_index"   : int,
                "tag"          : str,
                "raw_span"     : str,     # unnormalized text
                "gt_span"      : str,     # GT normalized text
                "left_context" : str,
                "right_context": str,
                "model_spans"  : {        # model → extracted span (or None)
                    "mt5"     : str or None,
                    "hints_ib": str or None,
                    "rb"      : str or None,
                },
                "cer": {                  # model → CER float
                    "mt5"     : float,
                    "hints_ib": float,
                    "rb"      : float,
                },
            },
            ...
        ]
    }
    """

    def __init__(
        self,
        filepath: str,
        model_keys: Optional[List[str]] = None,
        context_words: int = 3,
        skip_on_error: bool = True,
        verbose: bool = True,
    ):
        self.filepath = Path(filepath)
        self.model_keys = model_keys if model_keys is not None else MODEL_KEYS
        self.context_words = context_words
        self.skip_on_error = skip_on_error
        self.verbose = verbose
        self._data: List[Dict[str, Any]] = []
        self._skipped_sentences: List[Dict[str, Any]] = []  # Track skipped sentences
        self._load()

    # ------------------------------------------------------------------ #

    def _load(self):
        """Load and process JSONL file with error handling"""
        processed_count = 0
        skipped_count = 0
        error_details = []
        
        with open(self.filepath, encoding='utf-8') as f:
            for line_no, raw_line in enumerate(f, start=1):
                raw_line = raw_line.strip()
                if not raw_line:
                    continue
                
                try:
                    obj = json.loads(raw_line)
                    processed_item = self._process(obj, sentence_id=line_no - 1)
                    if processed_item is not None:
                        self._data.append(processed_item)
                        processed_count += 1
                    else:
                        skipped_count += 1
                        
                except json.JSONDecodeError as e:
                    error_msg = f"Bad JSON on line {line_no}: {e}"
                    error_details.append(error_msg)
                    if self.verbose:
                        print(f"Warning: {error_msg}")
                    skipped_count += 1
                    
                except Exception as e:
                    error_msg = f"Error processing line {line_no}: {str(e)}"
                    error_details.append(error_msg)
                    if self.verbose:
                        print(f"Warning: {error_msg}")
                    skipped_count += 1
        
        if self.verbose:
            print(f"\n{'='*60}")
            print(f"Dataset Loading Summary:")
            print(f"  Successfully loaded: {processed_count} sentences")
            print(f"  Skipped: {skipped_count} sentences")
            print(f"  Total processed: {processed_count + skipped_count} lines")
            print(f"{'='*60}\n")
        
        # Store error details for debugging
        self._error_details = error_details

    def _process(self, obj: Dict[str, Any], sentence_id: int) -> Optional[Dict[str, Any]]:
        """Process a single JSON object, return None if problematic"""
        try:
            tagged = obj.get('tagged_output', '')
            norm = obj.get('norm', '')
            
            # Skip if empty
            if not tagged or not norm:
                if self.verbose:
                    print(f"Warning: Sentence {sentence_id} missing tagged_output or norm - skipping")
                return None

            # 1. Plain (unnormalized) sentence
            unnorm_sentence = _strip_tags(tagged)

            # 2. Span lists
            raw_spans = extract_raw_spans(tagged, self.context_words)
            gt_spans = extract_gt_spans(norm)

            # Skip if span counts don't match
            if len(raw_spans) != len(gt_spans):
                if self.verbose:
                    print(f"Warning: Sentence {sentence_id}: {len(raw_spans)} raw spans vs "
                          f"{len(gt_spans)} GT spans - skipping")
                    print(f"  Tagged: {tagged[:100]}...")
                    print(f"  Norm: {norm[:100]}...")
                return None

            # 3. For every span, extract model span + CER
            spans = []
            for span_idx, (rs, gt) in enumerate(zip(raw_spans, gt_spans)):
                model_spans: Dict[str, Optional[str]] = {}
                cer: Dict[str, float] = {}

                for mk in self.model_keys:
                    model_text = obj.get(mk, '')
                    try:
                        extracted = _find_between(
                            model_text,
                            rs['left_context'],
                            rs['right_context'],
                        )
                    except Exception as e:
                        if self.verbose:
                            print(f"Warning: Error extracting span for model {mk}: {e}")
                        extracted = None
                    
                    model_spans[mk] = extracted
                    try:
                        cer[mk] = compute_cer(extracted or '', gt)
                    except Exception as e:
                        if self.verbose:
                            print(f"Warning: Error computing CER for {mk}: {e}")
                        cer[mk] = 1.0

                spans.append({
                    'span_index': span_idx,
                    'tag': rs['tag'],
                    'raw_span': rs['raw_span'],
                    'gt_span': gt,
                    'left_context': rs['left_context'],
                    'right_context': rs['right_context'],
                    'model_spans': model_spans,
                    'cer': cer,
                })

            return {
                'sentence_id': sentence_id,
                'unnorm_sentence': unnorm_sentence,
                'spans': spans,
            }
            
        except Exception as e:
            if self.verbose:
                print(f"Error processing sentence {sentence_id}: {str(e)[:200]}")
            return None

    # ------------------------------------------------------------------ #
    # Dataset interface

    def __len__(self) -> int:
        return len(self._data)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        return self._data[idx]

    # ------------------------------------------------------------------ #
    # Helpers

    def summary(self) -> Dict[str, float]:
        """Mean CER per model across every span in the dataset."""
        if not self._data:
            return {}
        
        totals: Dict[str, float] = {}
        counts: Dict[str, int] = {}
        for sent in self._data:
            for span in sent['spans']:
                for mk, c in span['cer'].items():
                    totals[mk] = totals.get(mk, 0.0) + c
                    counts[mk] = counts.get(mk, 0) + 1
        return {mk: totals[mk] / counts[mk] for mk in totals}

    def iter_spans(self):
        """Flat iterator: yields (sentence_id, span_dict) for every span."""
        for sent in self._data:
            for span in sent['spans']:
                yield sent['sentence_id'], span
    
    def get_tag_statistics(self) -> Dict[str, Dict[str, Any]]:
        """Get CER statistics per tag type"""
        tag_stats: Dict[str, Dict[str, Any]] = {}
        
        for sent in self._data:
            for span in sent['spans']:
                tag = span['tag']
                if tag not in tag_stats:
                    tag_stats[tag] = {
                        'count': 0,
                        'total_cer': {mk: 0.0 for mk in self.model_keys},
                        'cer_values': {mk: [] for mk in self.model_keys}
                    }
                
                tag_stats[tag]['count'] += 1
                for mk in self.model_keys:
                    cer_val = span['cer'][mk]
                    tag_stats[tag]['total_cer'][mk] += cer_val
                    tag_stats[tag]['cer_values'][mk].append(cer_val)
        
        # Compute averages
        for tag in tag_stats:
            count = tag_stats[tag]['count']
            for mk in self.model_keys:
                tag_stats[tag][f'avg_cer_{mk}'] = tag_stats[tag]['total_cer'][mk] / count if count > 0 else 0.0
        
        return tag_stats
    
    def get_problematic_sentences(self) -> List[Dict[str, Any]]:
        """Return list of skipped sentences for debugging"""
        return self._skipped_sentences


# ─────────────────────────────────────────────────────────────────────────────
# Collate function for DataLoader
# ─────────────────────────────────────────────────────────────────────────────
def collate_span_batch(batch, max_length=512):
    """
    Collate function that tokenizes sentences and batches spans with scores.
    
    Args:
        batch: List of dict items from SpanCERDataset.__getitem__
        tokenizer: HuggingFace tokenizer instance
        max_length: Maximum sequence length for tokenization
        
    Returns:
        Dictionary with:
        - tokenized: Tokenized inputs (input_ids, attention_mask, etc.)
        - sentences: List of original sentence texts
        - sentence_ids: List of sentence IDs
        - spans: List of all spans with metadata
        - model_scores: Dict of model names to CER score tensors
        - num_spans_per_sentence: List of span counts
    """
    if not batch:
        return {}
    
    # Extract sentences for tokenization
    sentences = [item['unnorm_sentence'] for item in batch]
    sentence_ids = [item['sentence_id'] for item in batch]
    num_spans_per_sentence = [len(item['spans']) for item in batch]
    
    # Tokenize all sentences in the batch
    tokenized = tokenizer(
        sentences,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors='pt'
    )
    
    # Flatten all spans from the batch
    all_spans = []
    span_to_sentence_idx = []
    
    for sent_idx, item in enumerate(batch):
        for span in item['spans']:
            all_spans.append({
                'span_index': span['span_index'],
                'tag': span['tag'],
                'raw_span': span['raw_span'],
                'gt_span': span['gt_span'],
                'left_context': span['left_context'],
                'right_context': span['right_context'],
                'model_spans': span['model_spans'],
                'cer': span['cer'],
            })
            span_to_sentence_idx.append(sent_idx)
    
    # Convert CER scores to tensors
    import torch
    model_keys = list(all_spans[0]['cer'].keys()) if all_spans else ['mt5', 'hints_ib', 'rb']
    model_scores = {mk: torch.tensor([s['cer'][mk] for s in all_spans], dtype=torch.float32) 
                    for mk in model_keys}
    
    # Extract span texts for each model
    model_extracted_spans = {mk: [s['model_spans'][mk] for s in all_spans] 
                              for mk in model_keys}
    
    return {
        # Tokenized outputs (ready for model)
        'tokenized': tokenized,
        
        # Original sentences
        'sentences': sentences,
        'sentence_ids': sentence_ids,
        'num_spans_per_sentence': num_spans_per_sentence,
        
        # Span information
        'spans': all_spans,
        'span_tags': [s['tag'] for s in all_spans],
        'span_raw_texts': [s['raw_span'] for s in all_spans],
        'span_gt_texts': [s['gt_span'] for s in all_spans],
        'span_left_contexts': [s['left_context'] for s in all_spans],
        'span_right_contexts': [s['right_context'] for s in all_spans],
        
        # Model scores and extracted spans
        'model_scores': model_scores,
        'model_extracted_spans': model_extracted_spans,
        
        # Mapping
        'span_to_sentence_idx': span_to_sentence_idx,
        'span_to_sentence_id': [sentence_ids[idx] for idx in span_to_sentence_idx],
    }
def collate_span_batch_with_tokens(batch, max_length=512):
    """
    Collate function that tokenizes sentences and returns structured output with all spans and model outputs.
    
    Args:
        batch: List of dict items from SpanCERDataset.__getitem__
        tokenizer: HuggingFace tokenizer instance (MT5Tokenizer)
        max_length: Maximum sequence length for tokenization
        
    Returns:
        Dictionary containing:
        - input_ids: Tensor of tokenized input IDs for all sentences [batch_size, seq_len]
        - attention_mask: Tensor of attention masks [batch_size, seq_len]
        - sentences: List of original unnormalized sentences
        - sentence_ids: List of sentence IDs
        - all_spans: List of all spans with their details (flattened across batch)
        - num_spans_per_sentence: List of span counts per sentence
        - model_scores: Dict of model names to CER score tensors
        - span_to_sentence_map: List mapping each span to its sentence index
    """
    if not batch:
        return {}
    
    # Extract sentences for tokenization
    sentences = [item['unnorm_sentence'] for item in batch]
    sentence_ids = [item['sentence_id'] for item in batch]
    num_spans_per_sentence = [len(item['spans']) for item in batch]
    
    # Tokenize all sentences in the batch
    tokenized = tokenizer(
        sentences,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors='pt'
    )
    
    # Flatten all spans from the batch
    all_spans = []
    span_to_sentence_idx = []
    
    for sent_idx, item in enumerate(batch):
        for span in item['spans']:
            all_spans.append({
                'span_index': span['span_index'],
                'tag': span['tag'],
                'raw_span': span['raw_span'],
                'gt_span': span['gt_span'],
                'left_context': span['left_context'],
                'right_context': span['right_context'],
                'mt5_output': span['model_spans'].get('mt5', ''),
                'hints_ib_output': span['model_spans'].get('hints_ib', ''),
                'rb_output': span['model_spans'].get('rb', ''),
                'mt5_cer': span['cer'].get('mt5', 1.0),
                'hints_ib_cer': span['cer'].get('hints_ib', 1.0),
                'rb_cer': span['cer'].get('rb', 1.0),
            })
            span_to_sentence_idx.append(sent_idx)
    
    # Convert CER scores to tensors
    import torch
    model_scores = {
        'mt5': torch.tensor([s['mt5_cer'] for s in all_spans], dtype=torch.float32),
        'hints_ib': torch.tensor([s['hints_ib_cer'] for s in all_spans], dtype=torch.float32),
        'rb': torch.tensor([s['rb_cer'] for s in all_spans], dtype=torch.float32),
    }
    
    return {
        # Tokenized outputs (ready for model)
        'input_ids': tokenized['input_ids'],
        'attention_mask': tokenized['attention_mask'],
        
        # Original sentences
        'sentences': sentences,
        'sentence_ids': sentence_ids,
        'num_spans_per_sentence': num_spans_per_sentence,
        
        # All spans flattened
        'all_spans': all_spans,
        
        # Model scores as tensors
        'model_scores': model_scores,
        
        # Mapping
        'span_to_sentence_idx': span_to_sentence_idx,
    }


In [ ]:
from torch.utils.data import random_split
import torch
dataset=SpanCERDataset("/home/sakshamt/SPIRE_TN/DATASET/Dataset_ver.2/lm_kan_ver2.txt")
dataset_size = len(dataset)
train_size = int(0.85 * dataset_size)
val_size = dataset_size - train_size

# Split the dataset
train_dataset, val_dataset = random_split(
    dataset, 
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)  # Fixed seed for reproducibility
)

In [ ]:
train_dataloader=DataLoader(train_dataset,batch_size=16,collate_fn=collate_span_batch_with_tokens)
val_dataloader=DataLoader(val_dataset,batch_size=16,collate_fn=collate_span_batch_with_tokens)

In [ ]:
# Decode and inspect the batch
for batch_idx, batch in enumerate(train_dataloader):
    print(f"\n{'='*60}")
    print(f"Batch {batch_idx + 1}")
    print(f"{'='*60}")
    print(f"  input_ids shape: {batch['input_ids'].shape}")
    print(f"  attention_mask shape: {batch['attention_mask'].shape}")
    print(f"  Number of sentences: {len(batch['sentences'])}")
    print(f"  Total spans: {len(batch['all_spans'])}")
    
    # Decode first sentence
    decoded = tokenizer.decode(batch['input_ids'][0], skip_special_tokens=True)
    print(f"\n📝 First Sentence (ID: {batch['sentence_ids'][0]}):")
    print(f"  {decoded}")
    
    # Get all spans for the first sentence
    num_spans_first_sentence = batch['num_spans_per_sentence'][0]
    first_sentence_spans = batch['all_spans'][:num_spans_first_sentence]
    
    print(f"\n🔍 All Spans for First Sentence ({len(first_sentence_spans)} spans):")
    print("-" * 60)
    
    for span_idx, span in enumerate(first_sentence_spans):
        print(f"\nSpan {span_idx + 1} (Tag: {span['tag']}):")
        print(f"  Raw text:     '{span['raw_span']}'")
        print(f"  GT text:      '{span['gt_span']}'")
        print(f"  Left context: '{span['left_context']}'")
        print(f"  Right context:'{span['right_context']}'")
        print(f"\n  Model Outputs:")
        print(f"    MT5:       '{span['mt5_output']}' (CER: {span['mt5_cer']:.4f})")
        print(f"    hints_ib:  '{span['hints_ib_output']}' (CER: {span['hints_ib_cer']:.4f})")
        print(f"    rb:        '{span['rb_output']}' (CER: {span['rb_cer']:.4f})")
        print("-" * 40)
    
    break  # Only show first batch

In [ ]:
import torch.nn.functional as F
from transformers import MT5EncoderModel, MT5Tokenizer
from torch.utils.data import DataLoader
from typing import Dict, List, Optional, Tuple
from tqdm import tqdm
import time
import torch
import torch.nn as nn
import os
import json
import shutil
from pathlib import Path
import pandas as pd
# ─────────────────────────────────────────────────────────────────────────────
# Model Definition (Classification with 3 outputs)
# ─────────────────────────────────────────────────────────────────────────────

class MT5SpanSelector(nn.Module):
    """
    MT5 Encoder model that predicts the best model for normalizing each span.
    
    Logit indices:
        0: RB model
        1: MT5 model  
        2: Hints_IB model
    
    Outputs logits for each model (classification, not regression).
    """
    
    def __init__(
        self,
        model_name: str = "google/mt5-small",
        hidden_size: int = 512,
        dropout: float = 0.1,
        num_models: int = 3
    ):
        super().__init__()
        
        # Load pretrained MT5 encoder
        self.encoder = MT5EncoderModel.from_pretrained(model_name)
        self.hidden_size = hidden_size
        
        # Projection layers
        self.span_projection = nn.Linear(hidden_size, hidden_size)
        
        # Separate projections for different span sources
        self.rb_proj = nn.Linear(hidden_size, hidden_size)
        self.mt5_proj = nn.Linear(hidden_size, hidden_size)
        self.hints_proj = nn.Linear(hidden_size, hidden_size)
        self.raw_proj = nn.Linear(hidden_size, hidden_size)
        
        # Combine all representations
        self.combine_layer = nn.Sequential(
            nn.Linear(hidden_size * 5, hidden_size * 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size * 2, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        # Classification head (outputs logits for RB, MT5, Hints_IB)
        self.classifier = nn.Linear(hidden_size, num_models)
        
        self.model_names = ['rb', 'mt5', 'hints_ib']
        self.model_to_idx = {'rb': 0, 'mt5': 1, 'hints_ib': 2}
        
    def encode_text(self, texts: List[str], tokenizer, max_length: int = 128) -> torch.Tensor:
        """Encode texts and return mean-pooled representations."""
        if not texts:
            return torch.zeros((0, self.hidden_size), device=self.encoder.device)
        
        encoded = tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors='pt'
        )
        
        encoded = {k: v.to(self.encoder.device) for k, v in encoded.items()}
        outputs = self.encoder(**encoded)
        
        # Mean pooling
        attention_mask = encoded['attention_mask'].unsqueeze(-1)
        embeddings = outputs.last_hidden_state
        masked_embeddings = embeddings * attention_mask
        summed = masked_embeddings.sum(dim=1)
        counts = attention_mask.sum(dim=1)
        mean_pooled = summed / counts
        
        return mean_pooled
    
    def forward(
        self,
        batch: Dict,
        tokenizer,
    ) -> Dict[str, torch.Tensor]:
        """
        Forward pass - predicts logits for each model.
        
        Returns:
            logits: [total_spans, 3] logits for RB, MT5, Hints_IB
        """
        sentences = batch['sentences']
        all_spans = batch['all_spans']
        span_to_sentence_idx = batch['span_to_sentence_idx']
        
        # Encode sentences
        sentence_embeddings = self.encode_text(sentences, tokenizer)
        
        # Prepare span texts
        rb_spans = []
        mt5_spans = []
        hints_spans = []
        raw_spans = []
        span_sentence_embeddings = []
        
        for span_idx, span in enumerate(all_spans):
            sent_idx = span_to_sentence_idx[span_idx]
            sent_emb = sentence_embeddings[sent_idx].unsqueeze(0)
            
            rb_spans.append(span.get('rb_output', '') or '')
            mt5_spans.append(span.get('mt5_output', '') or '')
            hints_spans.append(span.get('hints_ib_output', '') or '')
            raw_spans.append(span.get('raw_span', '') or '')
            span_sentence_embeddings.append(sent_emb)
        
        # Encode all span texts
        rb_embeddings = self.encode_text(rb_spans, tokenizer, max_length=32)
        mt5_embeddings = self.encode_text(mt5_spans, tokenizer, max_length=32)
        hints_embeddings = self.encode_text(hints_spans, tokenizer, max_length=32)
        raw_embeddings = self.encode_text(raw_spans, tokenizer, max_length=32)
        
        # Stack sentence embeddings
        sentence_embeddings_stack = torch.cat(span_sentence_embeddings, dim=0)
        
        # Apply projections
        rb_proj = self.rb_proj(rb_embeddings)
        mt5_proj = self.mt5_proj(mt5_embeddings)
        hints_proj = self.hints_proj(hints_embeddings)
        raw_proj = self.raw_proj(raw_embeddings)
        sent_proj = self.span_projection(sentence_embeddings_stack)
        
        # Concatenate
        combined = torch.cat([
            sent_proj, raw_proj, rb_proj, mt5_proj, hints_proj
        ], dim=1)
        
        # Combine features
        combined_features = self.combine_layer(combined)
        
        # Classification logits
        logits = self.classifier(combined_features)
        
        return {
            'logits': logits,  # [num_spans, 3]
        }


# ─────────────────────────────────────────────────────────────────────────────
# Cross Entropy Loss with Averaging over Tied Models
# ─────────────────────────────────────────────────────────────────────────────

def cross_entropy_with_ties(
    logits: torch.Tensor,  # [num_spans, 3]
    spans: List[Dict],
    device: torch.device,
    temperature: float = 1.0
) -> Tuple[torch.Tensor, Dict]:
    """
    Cross entropy loss that averages over all tied models.
    
    For a span with ties (multiple models have same minimum CER):
    1. Average the logits of all tied models
    2. Use this average as the "target" logit
    3. Apply cross entropy loss to make all tied models have high probability
    
    This is equivalent to having a soft target distribution where
    tied models share probability mass equally.
    """
    batch_size = logits.shape[0]
    total_loss = 0.0
    
    # For metrics
    num_ties = 0
    num_single = 0
    avg_confidence = 0.0
    
    for i in range(batch_size):
        span = spans[i]
        
        # Get actual CERs
        actual_cers = [
            span.get('rb_cer', 1.0),
            span.get('mt5_cer', 1.0),
            span.get('hints_ib_cer', 1.0)
        ]
        
        # Find minimum CER
        min_cer = min(actual_cers)
        
        # Find all indices with minimum CER (ties)
        best_indices = [j for j, cer in enumerate(actual_cers) if cer == min_cer]
        
        # Create target distribution
        target = torch.zeros(3, device=device)
        prob_per_tied = 1.0 / len(best_indices)
        for idx in best_indices:
            target[idx] = prob_per_tied
        
        # Get logits for this span
        span_logits = logits[i] / temperature
        
        # Cross entropy loss with soft targets
        # Equivalent to: -sum(target * log_softmax(logits))
        log_probs = F.log_softmax(span_logits, dim=0)
        loss_i = -torch.sum(target * log_probs)
        
        total_loss += loss_i
        
        # Track metrics
        if len(best_indices) > 1:
            num_ties += 1
        else:
            num_single += 1
        
        # Calculate confidence (probability assigned to best models)
        probs = F.softmax(span_logits, dim=0)
        confidence = sum(probs[idx].item() for idx in best_indices)
        avg_confidence += confidence
    
    total_loss = total_loss / batch_size
    avg_confidence = avg_confidence / batch_size
    
    metrics = {
        'loss': total_loss.item(),
        'num_ties': num_ties,
        'num_single_winners': num_single,
        'tie_percentage': (num_ties / batch_size) * 100 if batch_size > 0 else 0,
        'avg_confidence': avg_confidence
    }
    
    return total_loss, metrics


# ─────────────────────────────────────────────────────────────────────────────
# Model Checkpoint Manager - Stores ALL epochs with metadata
# ─────────────────────────────────────────────────────────────────────────────

class AllEpochsCheckpointManager:
    """
    Manages saving ALL model checkpoints with comprehensive metadata.
    Stores every epoch's model and tracks all validation metrics.
    """
    def __init__(self, save_dir: str):
        """
        Args:
            save_dir: Directory to save model checkpoints
        """
        self.save_dir = Path(save_dir)
        self.save_dir.mkdir(parents=True, exist_ok=True)
        
        # Create models directory (stores all epoch models)
        self.models_dir = self.save_dir / "all_models"
        self.models_dir.mkdir(exist_ok=True)
        
        # Metadata file to store all validation metrics per epoch
        self.metadata_file = self.save_dir / "training_metadata.json"
        
        # Initialize or load existing metadata
        self.metadata = self._load_or_create_metadata()
        
    def _load_or_create_metadata(self) -> Dict:
        """Load existing metadata or create new one."""
        if self.metadata_file.exists():
            with open(self.metadata_file, 'r') as f:
                return json.load(f)
        else:
            return {
                'training_info': {
                    'start_time': time.time(),
                    'total_epochs': 0,
                    'best_epoch': None,
                    'best_val_loss': float('inf'),
                    'best_val_accuracy': 0.0
                },
                'epochs': {}  # epoch_number -> metrics
            }
    
    def _save_metadata(self):
        """Save metadata to JSON file."""
        self.metadata['last_updated'] = time.time()
        with open(self.metadata_file, 'w') as f:
            json.dump(self.metadata, f, indent=2)
    
    def save_epoch(
        self, 
        model: nn.Module, 
        epoch: int, 
        train_metrics: Dict,
        val_metrics: Dict,
        optimizer_state: Optional[Dict] = None,
        scheduler_state: Optional[Dict] = None
    ):
        """
        Save model checkpoint for a specific epoch with all metrics.
        
        Args:
            model: The model to save
            epoch: Epoch number
            train_metrics: Training metrics for this epoch
            val_metrics: Validation metrics for this epoch
            optimizer_state: Optimizer state dict (optional)
            scheduler_state: Scheduler state dict (optional)
        """
        # Save model checkpoint
        model_path = self.models_dir / f"epoch_{epoch:04d}.pt"
        
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'train_metrics': train_metrics,
            'val_metrics': val_metrics,
            'timestamp': time.time()
        }
        
        # Add optimizer and scheduler states if provided
        if optimizer_state is not None:
            checkpoint['optimizer_state_dict'] = optimizer_state
        if scheduler_state is not None:
            checkpoint['scheduler_state_dict'] = scheduler_state
        
        # Save checkpoint
        torch.save(checkpoint, model_path)
        
        # Update metadata
        self.metadata['epochs'][str(epoch)] = {
            'epoch': epoch,
            'train_loss': train_metrics.get('loss', None),
            'train_confidence': train_metrics.get('avg_confidence', None),
            'train_ties_percentage': train_metrics.get('tie_percentage', None),
            'val_loss': val_metrics.get('val_loss', None),
            'val_accuracy': val_metrics.get('overall_accuracy', None),
            'val_single_accuracy': val_metrics.get('single_winner_accuracy', None),
            'val_tie_accuracy': val_metrics.get('tie_accuracy', None),
            'num_tie_spans': val_metrics.get('num_tie_spans', 0),
            'num_single_spans': val_metrics.get('num_single_spans', 0),
            'total_spans': val_metrics.get('total_spans', 0),
            'model_file': f"all_models/epoch_{epoch:04d}.pt",
            'timestamp': time.time()
        }
        
        # Update best epoch info
        current_val_loss = val_metrics.get('val_loss', float('inf'))
        current_val_acc = val_metrics.get('overall_accuracy', 0)
        
        if current_val_loss < self.metadata['training_info']['best_val_loss']:
            self.metadata['training_info']['best_val_loss'] = current_val_loss
            self.metadata['training_info']['best_epoch'] = epoch
            self.metadata['training_info']['best_val_accuracy'] = current_val_acc
            
            # Also save a symlink or copy as best_model.pt for easy access
            best_model_path = self.save_dir / "best_model.pt"
            if best_model_path.exists():
                best_model_path.unlink()
            # Create a symlink to the best model
            try:
                best_model_path.symlink_to(model_path.relative_to(self.save_dir))
            except:
                # If symlink fails (Windows without admin), copy instead
                shutil.copy(model_path, best_model_path)
        
        self.metadata['training_info']['total_epochs'] = max(
            self.metadata['training_info']['total_epochs'], epoch
        )
        
        # Save metadata
        self._save_metadata()
        
        return model_path
    
    def get_epoch_model_path(self, epoch: int) -> Path:
        """Get path to a specific epoch's model."""
        return self.models_dir / f"epoch_{epoch:04d}.pt"
    
    def load_epoch(self, model: nn.Module, epoch: int, device: torch.device, load_optimizer: bool = False):
        """
        Load a specific epoch's model.
        
        Args:
            model: The model to load weights into
            epoch: Epoch number to load
            device: Device to load model to
            load_optimizer: If True, also return optimizer and scheduler states
        """
        model_path = self.get_epoch_model_path(epoch)
        if not model_path.exists():
            print(f"❌ Model for epoch {epoch} not found at {model_path}")
            return None
        
        checkpoint = torch.load(model_path, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        
        print(f"✅ Loaded model from epoch {epoch}")
        print(f"   Train Loss: {checkpoint['train_metrics'].get('loss', 'N/A'):.4f}")
        print(f"   Val Loss: {checkpoint['val_metrics'].get('val_loss', 'N/A'):.4f}")
        print(f"   Val Accuracy: {checkpoint['val_metrics'].get('overall_accuracy', 'N/A'):.4f}")
        
        if load_optimizer:
            return checkpoint
        return checkpoint
    
    def load_best_model(self, model: nn.Module, device: torch.device):
        """Load the best model based on validation loss."""
        best_epoch = self.metadata['training_info']['best_epoch']
        if best_epoch is None:
            print("❌ No best epoch found in metadata")
            return None
        
        print(f"🏆 Loading best model from epoch {best_epoch}")
        return self.load_epoch(model, best_epoch, device)
    
    def get_all_epochs_summary(self) -> pd.DataFrame:
        """Get a pandas DataFrame summary of all epochs."""
        import pandas as pd
        
        epochs_data = []
        for epoch_str, data in self.metadata['epochs'].items():
            epochs_data.append({
                'Epoch': data['epoch'],
                'Train Loss': data['train_loss'],
                'Val Loss': data['val_loss'],
                'Val Accuracy': data['val_accuracy'],
                'Single Acc': data['val_single_accuracy'],
                'Tie Acc': data['val_tie_accuracy'],
                'Tie Spans %': (data['num_tie_spans'] / data['total_spans'] * 100) if data['total_spans'] > 0 else 0
            })
        
        df = pd.DataFrame(epochs_data)
        df = df.sort_values('Epoch')
        return df
    
    def print_summary(self):
        """Print a summary of all saved epochs."""
        print("\n" + "="*80)
        print("TRAINING SUMMARY - ALL EPOCHS")
        print("="*80)
        
        if not self.metadata['epochs']:
            print("No epochs saved yet.")
            return
        
        print(f"\n📊 Total Epochs Saved: {len(self.metadata['epochs'])}")
        print(f"🏆 Best Epoch: {self.metadata['training_info']['best_epoch']}")
        print(f"   Best Val Loss: {self.metadata['training_info']['best_val_loss']:.6f}")
        print(f"   Best Val Accuracy: {self.metadata['training_info']['best_val_accuracy']:.4f}")
        
        print("\n📈 Per-Epoch Metrics:")
        print("-" * 80)
        print(f"{'Epoch':<8} {'Train Loss':<12} {'Val Loss':<12} {'Val Acc':<10} {'Single Acc':<12} {'Tie Acc':<10}")
        print("-" * 80)
        
        for epoch_str, data in sorted(self.metadata['epochs'].items(), key=lambda x: int(x[0])):
            print(f"{data['epoch']:<8} {data['train_loss']:<12.4f} {data['val_loss']:<12.4f} "
                  f"{data['val_accuracy']:<10.4f} {data['val_single_accuracy']:<12.4f} "
                  f"{data['val_tie_accuracy']:<10.4f}")
        
        print("="*80)
        
        # Print best models path
        print(f"\n💾 All models saved in: {self.models_dir}")
        print(f"📄 Metadata saved at: {self.metadata_file}")
        print(f"⭐ Best model symlink: {self.save_dir / 'best_model.pt'}")
    
    def get_training_history(self) -> Dict:
        """Get complete training history as a dictionary."""
        return self.metadata


# ─────────────────────────────────────────────────────────────────────────────
# Training Loop with Progress Bar
# ─────────────────────────────────────────────────────────────────────────────

class ProgressBar:
    """Custom progress bar for training visualization"""
    def __init__(self, total, desc="Training", unit="batch"):
        self.pbar = tqdm(total=total, desc=desc, unit=unit, 
                         bar_format='{l_bar}{bar:40}{r_bar}{bar:-10b}')
        self.start_time = time.time()
        
    def update(self, n=1, **kwargs):
        self.pbar.update(n)
        if kwargs:
            postfix_str = " | ".join([f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}" 
                                      for k, v in kwargs.items()])
            self.pbar.set_postfix_str(postfix_str)
    
    def close(self):
        self.pbar.close()
        elapsed = time.time() - self.start_time
        print(f"  ✓ Completed in {elapsed:.2f} seconds")


def train_epoch(
    model: MT5SpanSelector,
    dataloader: DataLoader,
    optimizer: torch.optim.Optimizer,
    tokenizer,
    device: torch.device,
    epoch_num: int,
    show_progress: bool = True
) -> Dict[str, float]:
    """Train for one epoch using cross entropy averaged over ties with progress bar."""
    model.train()
    total_loss = 0
    total_confidence = 0
    total_ties = 0
    num_batches = 0
    
    # Setup progress bar
    if show_progress:
        pbar = ProgressBar(len(dataloader), desc=f"Epoch {epoch_num} [Train]", unit="batch")
    
    for batch_idx, batch in enumerate(dataloader):
        # Move batch to device
        batch_on_device = {}
        for k, v in batch.items():
            if torch.is_tensor(v):
                batch_on_device[k] = v.to(device)
            else:
                batch_on_device[k] = v
        
        # Forward pass
        outputs = model(batch_on_device, tokenizer)
        logits = outputs['logits']
        
        # Compute loss - averages over tied models
        loss, metrics = cross_entropy_with_ties(logits, batch['all_spans'], device)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += metrics['loss']
        total_confidence += metrics['avg_confidence']
        total_ties += metrics['num_ties']
        num_batches += 1
        
        # Update progress bar
        if show_progress:
            pbar.update(1, 
                       Loss=f"{metrics['loss']:.4f}",
                       Conf=f"{metrics['avg_confidence']:.3f}",
                       Ties=f"{metrics['tie_percentage']:.1f}%")
        
        # Print every 10 batches if no progress bar
        if not show_progress and batch_idx % 10 == 0:
            print(f"  Batch {batch_idx}: Loss={metrics['loss']:.4f}, "
                  f"Ties={metrics['tie_percentage']:.1f}%, "
                  f"Confidence={metrics['avg_confidence']:.3f}")
    
    if show_progress:
        pbar.close()
    
    return {
        'loss': total_loss / num_batches,
        'avg_confidence': total_confidence / num_batches,
        'avg_ties_per_batch': total_ties / num_batches,
        'tie_percentage': (total_ties / num_batches) * 100 if num_batches > 0 else 0
    }


def evaluate_epoch(
    model: MT5SpanSelector,
    dataloader: DataLoader,
    tokenizer,
    device: torch.device,
    epoch_num: int,
    show_progress: bool = True
) -> Dict[str, float]:
    """
    Evaluate accuracy with special handling for ties.
    
    For spans with ties, we consider the prediction correct if it
    predicts ANY of the tied models as the best.
    """
    model.eval()
    total_correct = 0
    total_spans = 0
    total_tie_correct = 0
    total_tie_spans = 0
    total_single_correct = 0
    total_single_spans = 0
    
    # Setup progress bar
    if show_progress:
        pbar = ProgressBar(len(dataloader), desc=f"Epoch {epoch_num} [Eval]", unit="batch")
    
    with torch.no_grad():
        for batch_idx, batch in enumerate(dataloader):
            # Move batch to device
            batch_on_device = {}
            for k, v in batch.items():
                if torch.is_tensor(v):
                    batch_on_device[k] = v.to(device)
                else:
                    batch_on_device[k] = v
            
            outputs = model(batch_on_device, tokenizer)
            logits = outputs['logits']
            predictions = torch.argmax(logits, dim=1).cpu()
            
            for i, span in enumerate(batch['all_spans']):
                actual_cers = [
                    span.get('rb_cer', 1.0),
                    span.get('mt5_cer', 1.0),
                    span.get('hints_ib_cer', 1.0)
                ]
                
                # Find actual best models (lowest CER)
                min_cer = min(actual_cers)
                best_indices = [j for j, cer in enumerate(actual_cers) if cer == min_cer]
                
                # Check if prediction is in best_indices
                is_correct = predictions[i].item() in best_indices
                
                if is_correct:
                    total_correct += 1
                
                if len(best_indices) > 1:
                    total_tie_spans += 1
                    if is_correct:
                        total_tie_correct += 1
                else:
                    total_single_spans += 1
                    if is_correct:
                        total_single_correct += 1
                
                total_spans += 1
            
            if show_progress:
                acc_so_far = total_correct / total_spans if total_spans > 0 else 0
                pbar.update(1, Acc=f"{acc_so_far:.4f}")
    
    if show_progress:
        pbar.close()
    
    overall_acc = total_correct / total_spans if total_spans > 0 else 0
    
    return {
        'overall_accuracy': overall_acc,
        'single_winner_accuracy': total_single_correct / total_single_spans if total_single_spans > 0 else 0,
        'tie_accuracy': total_tie_correct / total_tie_spans if total_tie_spans > 0 else 0,
        'num_tie_spans': total_tie_spans,
        'num_single_spans': total_single_spans,
        'total_spans': total_spans,
        'val_loss': 1.0 - overall_acc  # Convert accuracy to loss for tracking
    }


# ─────────────────────────────────────────────────────────────────────────────
# Complete Training Pipeline - Stores ALL Epochs
# ─────────────────────────────────────────────────────────────────────────────

def train_model(
    model: MT5SpanSelector,
    train_dataloader: DataLoader,
    val_dataloader: DataLoader,
    tokenizer,
    device: torch.device,
    num_epochs: int = 5,
    learning_rate: float = 2e-5,
    show_progress: bool = True,
    save_dir: str = "./checkpoints",
    save_all_epochs: bool = True
):
    """
    Complete training pipeline that saves ALL epoch weights.
    
    Args:
        save_dir: Directory to save model checkpoints
        save_all_epochs: If True, saves every epoch (default: True)
    """
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.5)
    
    # Initialize checkpoint manager (stores ALL epochs)
    checkpoint_manager = AllEpochsCheckpointManager(save_dir)
    
    # Training history (also stored in checkpoint_manager)
    history = {
        'train_loss': [],
        'train_confidence': [],
        'val_accuracy': [],
        'val_loss': [],
        'val_single_acc': [],
        'val_tie_acc': [],
        'learning_rates': []
    }
    
    print("\n" + "="*80)
    print("Starting Training".center(80))
    print("="*80)
    print(f"Device: {device}")
    print(f"Num Epochs: {num_epochs}")
    print(f"Learning Rate: {learning_rate}")
    print(f"Train Batches: {len(train_dataloader)}")
    print(f"Val Batches: {len(val_dataloader)}")
    print(f"Save Directory: {save_dir}")
    print(f"Saving ALL Epochs: {save_all_epochs}")
    print("="*80 + "\n")
    
    for epoch in range(num_epochs):
        print(f"\n{'─'*80}")
        print(f"Epoch {epoch + 1}/{num_epochs}")
        print(f"{'─'*80}")
        
        # Training
        train_metrics = train_epoch(
            model, train_dataloader, optimizer, tokenizer, device, 
            epoch_num=epoch+1, show_progress=show_progress
        )
        
        # Evaluation
        val_metrics = evaluate_epoch(
            model, val_dataloader, tokenizer, device, 
            epoch_num=epoch+1, show_progress=show_progress
        )
        
        # Update scheduler
        scheduler.step(val_metrics['val_loss'])
        current_lr = optimizer.param_groups[0]['lr']
        
        # Store history
        history['train_loss'].append(train_metrics['loss'])
        history['train_confidence'].append(train_metrics['avg_confidence'])
        history['val_accuracy'].append(val_metrics['overall_accuracy'])
        history['val_loss'].append(val_metrics['val_loss'])
        history['val_single_acc'].append(val_metrics['single_winner_accuracy'])
        history['val_tie_acc'].append(val_metrics['tie_accuracy'])
        history['learning_rates'].append(current_lr)
        
        # Save checkpoint for this epoch (ALWAYS save if save_all_epochs is True)
        if save_all_epochs:
            checkpoint_manager.save_epoch(
                model=model,
                epoch=epoch + 1,
                train_metrics=train_metrics,
                val_metrics=val_metrics,
                optimizer_state=optimizer.state_dict(),
                scheduler_state=scheduler.state_dict()
            )
            saved_msg = "✅ Saved"
        else:
            # Only save if better (optional)
            was_better = val_metrics['val_loss'] < checkpoint_manager.metadata['training_info']['best_val_loss']
            if was_better or epoch == 0:
                checkpoint_manager.save_epoch(
                    model=model,
                    epoch=epoch + 1,
                    train_metrics=train_metrics,
                    val_metrics=val_metrics,
                    optimizer_state=optimizer.state_dict(),
                    scheduler_state=scheduler.state_dict()
                )
                saved_msg = "✅ Saved (better)"
            else:
                saved_msg = "⏭️ Skipped (not better)"
        
        # Print epoch summary
        print(f"\n{'─'*40}")
        print(f"Epoch {epoch+1} Summary:")
        print(f"{'─'*40}")
        print(f"Training:")
        print(f"   Loss: {train_metrics['loss']:.4f}")
        print(f"   Confidence: {train_metrics['avg_confidence']:.3f}")
        print(f"   Ties: {train_metrics['tie_percentage']:.1f}%")
        print(f"\n Validation:")
        print(f"   Overall Accuracy: {val_metrics['overall_accuracy']:.4f}")
        print(f"   Validation Loss: {val_metrics['val_loss']:.4f}")
        print(f"   Single Winner Acc: {val_metrics['single_winner_accuracy']:.4f}")
        print(f"   Tie Acc: {val_metrics['tie_accuracy']:.4f}")
        print(f"   Tie Spans: {val_metrics['num_tie_spans']}/{val_metrics['total_spans']} ({val_metrics['num_tie_spans']/val_metrics['total_spans']*100:.1f}%)")
        print(f"\ Learning Rate: {current_lr:.2e}")
        print(f" Checkpoint: {saved_msg}")
        print(f"{'─'*40}")
    
    print("\n" + "="*80)
    print("Training Complete!".center(80))
    print("="*80)
    
    # Print final summary
    checkpoint_manager.print_summary()
    
    return history, checkpoint_manager


# ─────────────────────────────────────────────────────────────────────────────
# Inference
# ─────────────────────────────────────────────────────────────────────────────

@torch.no_grad()
def predict_best_model(
    model: MT5SpanSelector,
    batch: Dict,
    tokenizer,
    device: torch.device
) -> Dict[str, List]:
    """
    Predict which single model is best for each span.
    
    For spans with ties, the model will still predict ONE model,
    but the confidence scores will be distributed.
    """
    model.eval()
    
    batch_on_device = {}
    for k, v in batch.items():
        if torch.is_tensor(v):
            batch_on_device[k] = v.to(device)
        else:
            batch_on_device[k] = v
    
    outputs = model(batch_on_device, tokenizer)
    logits = outputs['logits']
    probs = F.softmax(logits, dim=1)
    predictions = torch.argmax(logits, dim=1)
    
    results = []
    for i in range(len(predictions)):
        results.append({
            'span_index': i,
            'predicted_best_model': model.model_names[predictions[i].item()],
            'predicted_best_index': predictions[i].item(),
            'probabilities': {
                'rb': probs[i, 0].item(),
                'mt5': probs[i, 1].item(),
                'hints_ib': probs[i, 2].item()
            },
            'confidence': probs[i, predictions[i]].item()
        })
    
    return {'predictions': results}


# ─────────────────────────────────────────────────────────────────────────────
# Helper function to load any checkpoint
# ─────────────────────────────────────────────────────────────────────────────

def load_checkpoint(model: MT5SpanSelector, checkpoint_path: str, device: torch.device):
    """Load a specific checkpoint file into the model."""
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"✅ Loaded checkpoint from {checkpoint_path}")
    print(f"   Epoch: {checkpoint['epoch']}")
    print(f"   Validation Loss: {checkpoint['val_metrics'].get('val_loss', 'N/A'):.4f}")
    print(f"   Validation Accuracy: {checkpoint['val_metrics'].get('overall_accuracy', 'N/A'):.4f}")
    return checkpoint


# ─────────────────────────────────────────────────────────────────────────────
# Example Usage
# ─────────────────────────────────────────────────────────────────────────────

def demonstrate_tie_handling():
    """
    Demonstrates how cross entropy with ties works.
    """
    print("\n" + "="*60)
    print("TIE HANDLING DEMONSTRATION")
    print("="*60)
    
    device = torch.device('cpu')
    
    # Example: Span with tie between RB (idx 0) and MT5 (idx 1)
    # Both have CER 0.0, Hints_IB has CER 0.5
    span = {
        'rb_cer': 0.0,
        'mt5_cer': 0.0,
        'hints_ib_cer': 0.5
    }
    spans = [span]
    
    # Case 1: Model predicts RB as best (high logit for RB)
    logits_case1 = torch.tensor([[2.0, 1.0, 0.5]])  # RB highest
    
    # Case 2: Model predicts MT5 as best
    logits_case2 = torch.tensor([[1.0, 2.0, 0.5]])  # MT5 highest
    
    # Case 3: Model distributes probability equally between RB and MT5
    logits_case3 = torch.tensor([[1.5, 1.5, 0.5]])  # RB and MT5 equal
    
    print(f"\nGround truth: RB and MT5 are BOTH correct (CER=0.0)")
    print(f"Hints_IB is wrong (CER=0.5)\n")
    
    results = []
    for i, logits in enumerate([logits_case1, logits_case2, logits_case3]):
        loss, metrics = cross_entropy_with_ties(logits, spans, device)
        probs = F.softmax(logits, dim=1)
        
        results.append({
            'case': i+1,
            'logits': logits[0].tolist(),
            'probs': [probs[0,0].item(), probs[0,1].item(), probs[0,2].item()],
            'loss': loss.item(),
            'confidence': metrics['avg_confidence']
        })
        
        print(f"Case {i+1}: Logits = {logits[0].tolist()}")
        print(f"  Probabilities: RB={probs[0,0]:.3f}, MT5={probs[0,1]:.3f}, Hints={probs[0,2]:.3f}")
        print(f"  Loss = {loss.item():.4f}")
        print(f"  Confidence in best models = {metrics['avg_confidence']:.3f}")
        print()
    
    # Find best case
    best_case = min(results, key=lambda x: x['loss'])
    print(f" Best case: Case {best_case['case']} (Loss={best_case['loss']:.4f})")
    print(f"  → Model learns to distribute probability between tied models")
    
    return results

In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
MODEL = MT5SpanSelector(
        model_name="google/mt5-small",
        hidden_size=512,
        dropout=0.1,
        num_models=3
    ).to(device)
    
print(f"Model initialized:")
print(f"  Total parameters: {sum(p.numel() for p in MODEL.parameters()):,}")
print(f"  Trainable parameters: {sum(p.numel() for p in MODEL.parameters() if p.requires_grad):,}")

In [ ]:
history, checkpoint_manager = train_model(
    model=MODEL,
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    tokenizer=tokenizer,
    device=device,
    num_epochs=10,
    learning_rate=2e-5,
    show_progress=True,
    save_dir="/home/sakshamt/SPIRE_TN/DATASET/Dataset_ver.2/lm_span_hi",  # Your target folder
    top_k=1,  # Keep top 5 models
    save_best_by="loss"  # Save based on validation loss
)

## Inference when no tags in text  are available

In [ ]:
"""
Fixed Span Identification + Reconstruction Pipeline
=====================================================
Preserves ALL input keys and adds lm_span_selector output.
"""

import re
import json
import difflib
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset


# ─────────────────────────────────────────────────────────────────────────────
# Tokenizer helpers
# ─────────────────────────────────────────────────────────────────────────────

def _tokenize(text: str) -> List[Tuple[str, int]]:
    tokens = []
    for m in re.finditer(r'\S+', text):
        tokens.append((m.group(), m.start()))
    return tokens


def _tokens_only(tok_pos: List[Tuple[str, int]]) -> List[str]:
    return [t for t, _ in tok_pos]


# ─────────────────────────────────────────────────────────────────────────────
# Alignment: find changed regions
# ─────────────────────────────────────────────────────────────────────────────

def align_and_find_spans(
    unnorm: str,
    model_out: str
) -> List[Tuple[int, int, str]]:
    unnorm_tok = _tokenize(unnorm)
    model_tok = _tokenize(model_out)

    un_words = _tokens_only(unnorm_tok)
    mo_words = _tokens_only(model_tok)

    matcher = difflib.SequenceMatcher(None, un_words, mo_words, autojunk=False)

    spans = []
    pending_un_start = None
    pending_un_end = None
    pending_mo_parts = []

    def flush():
        if pending_un_start is not None and pending_mo_parts:
            replacement = " ".join(pending_mo_parts)
            spans.append((pending_un_start, pending_un_end, replacement))

    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag == "equal":
            flush()
            pending_un_start = None
            pending_un_end = None
            pending_mo_parts = []
        else:
            if i1 < i2:
                first_start = unnorm_tok[i1][1]
                last_tok, last_start = unnorm_tok[i2 - 1]
                last_end = last_start + len(last_tok)

                if pending_un_start is None:
                    pending_un_start = first_start
                pending_un_end = last_end
            pending_mo_parts.extend(mo_words[j1:j2])

    flush()
    return spans


# ─────────────────────────────────────────────────────────────────────────────
# Vote spans across models
# ─────────────────────────────────────────────────────────────────────────────

def vote_spans(
    unnorm: str,
    model_outputs: Dict[str, str],
) -> List[Dict]:
    range_to_models: Dict[Tuple[int, int], Dict[str, str]] = {}

    for model_name, model_out in model_outputs.items():
        if not model_out or not model_out.strip():
            continue
        for start, end, replacement in align_and_find_spans(unnorm, model_out):
            key = (start, end)
            if key not in range_to_models:
                range_to_models[key] = {}
            range_to_models[key][model_name] = replacement

    spans = []
    for (start, end), model_map in range_to_models.items():
        unnorm_span = unnorm[start:end]
        if not unnorm_span.strip():
            continue

        replacements = list(model_map.values())
        consensus = len(set(replacements)) == 1 and len(model_map) == len(model_outputs)

        spans.append({
            "unnorm_span": unnorm_span,
            "start": start,
            "end": end,
            "model_spans": model_map,
            "vote_count": len(model_map),
            "consensus": consensus,
        })

    spans.sort(key=lambda s: s["start"])
    return _remove_overlaps(spans)


def _remove_overlaps(spans: List[Dict]) -> List[Dict]:
    result = []
    used_end = -1
    for sp in spans:
        if sp["start"] >= used_end:
            result.append(sp)
            used_end = sp["end"]
        else:
            last = result[-1]
            if sp["vote_count"] > last["vote_count"] or (
                sp["vote_count"] == last["vote_count"] and
                (sp["end"] - sp["start"]) > (last["end"] - last["start"])
            ):
                result[-1] = sp
                used_end = sp["end"]
    return result


# ─────────────────────────────────────────────────────────────────────────────
# Dataset - Dynamically detects ALL model keys
# ─────────────────────────────────────────────────────────────────────────────

class SpanIdentificationDataset_up(Dataset):
    def __init__(self, file_path: str, tokenizer=None, max_length: int = 512):
        self.file_path = file_path
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.data: List[Dict] = []
        self.model_keys: List[str] = []
        self._load()

    def _load(self):
        with open(self.file_path, encoding="utf-8") as f:
            for line_num, line in enumerate(f):
                line = line.strip()
                if not line:
                    continue
                try:
                    raw = json.loads(line)
                except json.JSONDecodeError as e:
                    print(f"[Line {line_num}] JSON error: {e}")
                    continue

                unnorm = raw.get("unnorm", "").strip()
                norm = raw.get("norm", "").strip()

                # Dynamically find all model output keys (exclude non-model keys)
                exclude_keys = {"unnorm", "norm", "tagged_ip", "tagged_op", "id", 
                               "cer_hints_ib", "cer_mt5", "cer_rb", "cer_indic_bart",
                               "cer_gemma_1b", "cer_mt5_large", "wer_hints_ib", 
                               "wer_mt5", "wer_rb", "wer_indic_bart", "wer_gemma_1b", 
                               "wer_mt5_large","gemma_1b","mt5_large","llama_1b","cer_llama_1b","wer_lama_1b"}
                
                model_outputs = {}
                for k, v in raw.items():
                    if k not in exclude_keys and isinstance(v, str) and len(v) > 10:
                        model_outputs[k] = v.strip()
                        if k not in self.model_keys:
                            self.model_keys.append(k)

                spans = vote_spans(unnorm, model_outputs)

                self.data.append({
                    "unnorm": unnorm,
                    "norm": norm,
                    "model_outputs": model_outputs,
                    "spans": spans,
                    "raw": raw,  # Store ALL original data
                })

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            "unnorm_text": item["unnorm"],
            "norm_text": item["norm"],
            "model_outputs": item["model_outputs"],
            "spans": item["spans"],
            "num_spans": len(item["spans"]),
        }

    def get_model_keys(self) -> List[str]:
        return self.model_keys


# ─────────────────────────────────────────────────────────────────────────────
# Batch builder for classifier
# ─────────────────────────────────────────────────────────────────────────────

def build_batch(
    unnorm: str,
    spans: List[Dict],
    model_outputs: Dict[str, str],
) -> Dict:
    all_spans = []
    for sp in spans:
        ms = sp["model_spans"]
        span_entry = {
            "raw_span": sp["unnorm_span"],
        }
        # Add all model outputs for this span
        for model_name, output in model_outputs.items():
            span_entry[f"{model_name}_output"] = ms.get(model_name, output)
        all_spans.append(span_entry)

    return {
        "sentences": [unnorm],
        "all_spans": all_spans,
        "span_to_sentence_idx": [0] * len(all_spans),
    }


# ─────────────────────────────────────────────────────────────────────────────
# Classifier
# ─────────────────────────────────────────────────────────────────────────────

@torch.no_grad()
def classify_spans(
    model,
    batch: Dict,
    tokenizer,
    device: torch.device,
    model_keys: List[str],
) -> List[Dict]:
    model.eval()
    batch_dev = {
        k: (v.to(device) if torch.is_tensor(v) else v)
        for k, v in batch.items()
    }

    outputs = model(batch_dev, tokenizer)
    logits = outputs["logits"]
    probs = F.softmax(logits, dim=1)
    preds = torch.argmax(logits, dim=1).cpu()

    results = []
    for i, span_entry in enumerate(batch["all_spans"]):
        pred_idx = preds[i].item()
        pred_name = model_keys[pred_idx] if pred_idx < len(model_keys) else model_keys[0]
        norm_span = span_entry.get(f"{pred_name}_output", "")

        probs_dict = {}
        for idx, key in enumerate(model_keys):
            if idx < probs.shape[1]:
                probs_dict[key] = round(probs[i, idx].item(), 4)

        results.append({
            "model_picked": pred_name,
            "normalised_span": norm_span,
            "confidence": round(probs[i, pred_idx].item(), 4),
            "probabilities": probs_dict,
        })
    return results


# ─────────────────────────────────────────────────────────────────────────────
# Reconstruction
# ─────────────────────────────────────────────────────────────────────────────

def reconstruct(
    unnorm: str,
    spans: List[Dict],
    preds: List[Dict],
) -> str:
    parts = []
    cursor = 0

    for sp, pred in zip(spans, preds):
        start = sp["start"]
        end = sp["end"]
        norm = pred["normalised_span"].strip()

        if start < cursor:
            continue

        parts.append(unnorm[cursor:start])
        parts.append(norm)
        cursor = end

    parts.append(unnorm[cursor:])
    result = "".join(parts)
    result = re.sub(r" +", " ", result).strip()
    return result


# ─────────────────────────────────────────────────────────────────────────────
# Full per-sentence inference
# ─────────────────────────────────────────────────────────────────────────────

def infer_sentence(
    model,
    tokenizer,
    device: torch.device,
    item: Dict,
    model_keys: List[str],
) -> Dict:
    unnorm = item["unnorm"]
    norm = item["norm"]
    spans = item["spans"]
    mo = item["model_outputs"]
    raw = item.get("raw", {})

    if not spans:
        result = dict(raw)
        result["lm_span_selector"] = unnorm
        result["spans_info"] = []
        return result

    batch = build_batch(unnorm, spans, mo)
    preds = classify_spans(model, batch, tokenizer, device, model_keys)
    recon = reconstruct(unnorm, spans, preds)

    # Preserve ALL original keys
    result = dict(raw)
    result["lm_span_selector"] = recon

    # Add span info
    enriched = []
    for sp, pred in zip(spans, preds):
        enriched.append({
            "unnorm_span": sp["unnorm_span"],
            "start_char": sp["start"],
            "end_char": sp["end"],
            "model_picked": pred["model_picked"],
            "normalised_span": pred["normalised_span"],
            "confidence": pred["confidence"],
            "probabilities": pred["probabilities"],
            "vote_count": sp["vote_count"],
            "consensus": sp["consensus"],
        })
    result["spans_info"] = enriched

    return result


# ─────────────────────────────────────────────────────────────────────────────
# Main inference loop
# ─────────────────────────────────────────────────────────────────────────────

def run_inference(
    model,
    tokenizer,
    device: torch.device,
    dataset: SpanIdentificationDataset_up,
    output_path: str,
    show_progress: bool = True,
) -> List[Dict]:
    results = []
    out_path = Path(output_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    total = len(dataset)
    model_keys = dataset.get_model_keys()

    model_counts = {k: 0 for k in model_keys}

    print(f"\nRunning inference on {total} sentences → {output_path}")
    print(f"Models found: {model_keys}\n")

    with open(out_path, "w", encoding="utf-8") as fout:
        for idx, item in enumerate(dataset.data):
            result = infer_sentence(model, tokenizer, device, item, model_keys)
            results.append(result)
            fout.write(json.dumps(result, ensure_ascii=False) + "\n")

            for sp in result.get("spans_info", []):
                model_counts[sp["model_picked"]] += 1

            if show_progress and (idx + 1) % 100 == 0:
                print(f"  [{idx+1}/{total}]")

    total_spans = sum(model_counts.values())
    print(f"Done. {total} sentences, {total_spans} spans processed.")
    if total_spans:
        print("  Model selection:")
        for m, c in sorted(model_counts.items(), key=lambda x: -x[1]):
            print(f"    {m:15s}: {c:5d} ({c/total_spans*100:.1f}%)")

    return results


# ─────────────────────────────────────────────────────────────────────────────
# Usage
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    # Demo without actual model
    def demo():
        with open("your_input.jsonl", "r") as f:
            sample = json.loads(f.readline())
        
        unnorm = sample["unnorm"]
        model_outputs = {}
        for k, v in sample.items():
            if k not in ["unnorm", "norm", "tagged_ip", "tagged_op", "id"]:
                if isinstance(v, str) and len(v) > 10:
                    model_outputs[k] = v
        
        spans = vote_spans(unnorm, model_outputs)
        
        print(f"Found {len(spans)} spans")
        for sp in spans:
            print(f"  '{sp['unnorm_span']}' -> votes={sp['vote_count']}")
        
        # Dummy predictions (always pick first model)
        dummy_preds = []
        for sp in spans:
            chosen = list(sp["model_spans"].keys())[0]
            dummy_preds.append({
                "model_picked": chosen,
                "normalised_span": sp["model_spans"][chosen],
            })
        
        recon = reconstruct(unnorm, spans, dummy_preds)
        
        result = dict(sample)
        result["lm_span_selector"] = recon
        result["spans_info"] = []
        
        print(f"\nReconstructed: {recon}")
    
    # demo()

In [ ]:
"""
Complete script to run inference and get reconstructed outputs
"""

# import torch
# from transformers import MT5Tokenizer
# from pathlib import Path

# Import your model class (adjust path as needed)
# from model import MT5SpanSelector

# Import the pipeline functions
# from span_pipeline_fixed import (
#     SpanIdentificationDataset,
#     run_inference,
#     # If you have your model class, import it
# )


    
INPUT_FILE = "/home/sakshamt/SPIRE_TN/DATASET/Dataset_ver.2/kan_w_g_mtl_llama.txt"      # Your input data
# CHECKPOINT_PATH = "path/to/model_checkpoint.pt"  # Your trained model
OUTPUT_FILE = "/home/sakshamt/SPIRE_TN/DATASET/Dataset_ver.2/kan_z_output.jsonl"
    # MODEL_NAME = "google/mt5-small"
# DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # ============================================================
    # 2. LOAD TOKENIZER
    # ============================================================
    # print(f"Loading tokenizer: {MODEL_NAME}")
    # tokenizer = MT5Tokenizer.from_pretrained(MODEL_NAME)
    
    # # ============================================================
    # 3. LOAD YOUR TRAINED MODEL
    # ============================================================
    # print(f"Loading model from: {CHECKPOINT_PATH}")
    
    # Import your model class (adjust according to your actual model file)
    # from your_model_file import MT5SpanSelector
    
    # Initialize model
    # model = MT5SpanSelector(
    #     model_name=MODEL_NAME,
    #     hidden_size=512,
    #     dropout=0.1,
    #     num_models=3
    # ).to(DEVICE)
    
 
    # ============================================================
    # print(f"\nLoading dataset from: {INPUT_FILE}")
dataset_up = SpanIdentificationDataset_up(
        file_path=INPUT_FILE,
        tokenizer=tokenizer,
        max_length=512
    )
    
    # Print dataset statistics
# stats = dataset_up.get_span_statistics()
# print(f"  Total sentences: {stats['total_examples']}")
# print(f"  Total spans identified: {stats['total_spans']}")
# print(f"  Avg spans per sentence: {stats['avg_spans_per_sentence']:.2f}")
# print(f"  Consensus rate: {stats['consensus_rate']*100:.1f}%")
    
#     # ============================================================
#     # 5. RUN INFERENCE
#     # ============================================================
results = run_inference(
        model=MODEL,
        tokenizer=tokenizer,
        device=device,
        dataset=dataset_up,
        output_path=OUTPUT_FILE,
        show_progress=True
    )
    
    # ============================================================
    # 6. VERIFY RESULTS
    # ============================================================
print(f"\n{'='*60}")
print("SAMPLE OUTPUT")
print('='*60)
    
if results:
        sample = results[0]
        print(f"Original: {sample['unnorm'][:100]}...")
        print(f"Ground Truth: {sample['norm'][:100]}...")
        print(f"Reconstructed: {sample['reconstructed'][:100]}...")
        print(f"\nSpans found: {len(sample['spans'])}")
        
        for span in sample['spans'][:3]:  # Show first 3 spans
            print(f"\n  Span: '{span['unnorm_span']}'")
            print(f"    Picked model: {span['model_picked']}")
            print(f"    Replaced with: '{span['normalised_span']}'")
            print(f"    Confidence: {span['confidence']:.4f}")
    
print(f"Done! Results saved to: {OUTPUT_FILE}")


